# 06 — Final SHAP and ADMET Interpretability

> Clean senior-review version. This notebook contains only the final, logically connected pipeline. 
> Exploratory/probing/failed scripts are intentionally excluded from the main workflow.

## Objective
Interpret only the final trained model and final RFE-selected features. ADMET-likeness is used as an early triage layer, not as a full PK/ADMET prediction platform.

In [ ]:
# !pip install -q pandas numpy matplotlib shap xgboost joblib

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

PROJECT_DIR = Path.cwd()
MODEL_DIR = PROJECT_DIR / "outputs" / "recA_chembl" / "final_model"
INTERP_DIR = PROJECT_DIR / "outputs" / "recA_chembl" / "interpretability"
INTERP_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILE = MODEL_DIR / "final_trained_model.joblib"
FEATURE_FILE = MODEL_DIR / "final_model_features.csv"

FDA_FINGERPRINT_FILE = PROJECT_DIR / "actual_reca_fda_prediction_research_files" / "data" / "fda_combined_fingerprints.csv"
FDA_DOCKING_FILE = PROJECT_DIR / "actual_reca_fda_prediction_research_files" / "data" / "fda_predictions_with_real_docking.csv"

## 1. Load final model and final features

In [ ]:
model = joblib.load(MODEL_FILE)
features = pd.read_csv(FEATURE_FILE)["feature"].tolist()

print("Loaded final model:", type(model))
print("Final features:", len(features))

## 2. Load FDA candidates and align selected features

In [ ]:
fp = pd.read_csv(FDA_FINGERPRINT_FILE)
dock = pd.read_csv(FDA_DOCKING_FILE)

if "cid" not in fp.columns:
    if "Name" in fp.columns:
        fp["cid"] = fp["Name"].astype(str).str.replace("CID", "", regex=False).astype(int)
    else:
        raise ValueError("FDA fingerprint file must contain either cid or Name.")

for feature in features:
    if feature not in fp.columns:
        fp[feature] = 0

candidate_df = dock.merge(fp[["cid"] + features], on="cid", how="inner")
X_candidates = candidate_df[features].apply(pd.to_numeric, errors="coerce").fillna(0)

candidate_df["final_model_probability_active"] = model.predict_proba(X_candidates)[:, 1]
candidate_df["final_model_predicted_class"] = np.where(
    candidate_df["final_model_probability_active"] >= 0.5,
    "active_like",
    "inactive_like",
)

print(candidate_df.shape)
candidate_df.head()

## 3. ADMET-likeness heuristic triage

In [ ]:
def add_admet_heuristics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    required_admet = ["xlogp", "tpsa", "hbd", "hba", "rotatable_bonds", "molecular_weight"]
    for col in required_admet:
        if col not in out.columns:
            out[col] = np.nan
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out["lipinski_violations"] = (
        (out["molecular_weight"] > 500).fillna(False).astype(int)
        + (out["xlogp"] > 5).fillna(False).astype(int)
        + (out["hbd"] > 5).fillna(False).astype(int)
        + (out["hba"] > 10).fillna(False).astype(int)
    )
    out["lipinski_pass"] = out["lipinski_violations"] <= 1
    out["veber_pass"] = (out["tpsa"] <= 140).fillna(False) & (out["rotatable_bonds"] <= 10).fillna(False)
    out["egan_like_pass"] = (out["xlogp"] <= 5.88).fillna(False) & (out["tpsa"] <= 131.6).fillna(False)

    out["admet_support_score"] = out[["lipinski_pass", "veber_pass", "egan_like_pass"]].astype(float).mean(axis=1)
    out["admet_support"] = pd.cut(
        out["admet_support_score"],
        bins=[-0.01, 0.32, 0.66, 1.01],
        labels=["High risk", "Borderline", "Supportive"],
    )

    if "binding_affinity_kcal_mol" in out.columns:
        affinity = pd.to_numeric(out["binding_affinity_kcal_mol"], errors="coerce")
        out["docking_support_norm"] = ((-affinity) - (-affinity).min()) / (((-affinity).max() - (-affinity).min()) + 1e-12)
    else:
        out["docking_support_norm"] = 0.0

    prob = out["final_model_probability_active"]
    out["ml_support_norm"] = (prob - prob.min()) / ((prob.max() - prob.min()) + 1e-12)

    out["integrated_support_score"] = (
        0.45 * out["docking_support_norm"]
        + 0.30 * out["ml_support_norm"]
        + 0.25 * out["admet_support_score"]
    )

    return out.sort_values("integrated_support_score", ascending=False).reset_index(drop=True)

candidate_df = add_admet_heuristics(candidate_df)
candidate_df.to_csv(INTERP_DIR / "candidate_admet_screening_final_model.csv", index=False)

candidate_df[[
    "cid", "final_model_probability_active", "admet_support",
    "integrated_support_score"
]].head(10)

## 4. SHAP explanation for the final model

In [ ]:
# SHAP works directly for tree-based models. For SVM/pipeline models, use KernelExplainer on a small background sample.
final_model_name = type(model).__name__

try:
    explainer = shap.Explainer(model, X_candidates)
    shap_values = explainer(X_candidates)
except Exception:
    background = shap.sample(X_candidates, min(50, len(X_candidates)), random_state=42)
    predict_fn = lambda data: model.predict_proba(pd.DataFrame(data, columns=features))[:, 1]
    explainer = shap.KernelExplainer(predict_fn, background)
    shap_values = explainer.shap_values(X_candidates, nsamples=100)

plt.figure()
shap.summary_plot(shap_values, X_candidates, show=False, max_display=20)
plt.tight_layout()
plt.savefig(INTERP_DIR / "final_model_shap_summary.png", dpi=220, bbox_inches="tight")
plt.close()

print(f"Saved SHAP summary: {INTERP_DIR / 'final_model_shap_summary.png'}")

## 5. Export concise interpretation table

In [ ]:
name_cols = [c for c in ["compound_name", "Name", "cid"] if c in candidate_df.columns]
summary_cols = name_cols + [
    "final_model_probability_active",
    "final_model_predicted_class",
    "admet_support",
    "admet_support_score",
    "integrated_support_score",
]
if "binding_affinity_kcal_mol" in candidate_df.columns:
    summary_cols.insert(len(name_cols) + 1, "binding_affinity_kcal_mol")

final_summary = candidate_df[summary_cols].copy()
final_summary.to_csv(INTERP_DIR / "final_candidate_prioritization_summary.csv", index=False)

final_summary.head(15)

## Final outputs
- `candidate_admet_screening_final_model.csv`
- `final_model_shap_summary.png`
- `final_candidate_prioritization_summary.csv`